# 011_create_data_flow_control_table
Creates `ctrl.data_flow_control` and `ctrl.source_type_registry`. Safe to re-run: a table is only
created if it is missing. Rows are added by `05_helper_scripts/050_populate_control_tables`.

Differences from the Databricks version:
- `depends_on_control_ids` holds a comma-separated list, because a fact usually waits for
  several tables (the Databricks column holds one id).
- `source_connection_name` names a folder in `CONNECTIONS` instead of a secret scope.
- `publish_name` is the file name the publish step writes for Power BI (Gold flows only).
- The incremental columns at the end drive the INCREMENTAL, PARTITION_REFRESH and SCD2 loaders
  (see `001_nb_utils`, Incremental loads).

In [ ]:
# Bootstrap: find the pipeline root and load the shared notebooks.
# run_notebook() plays the part of Databricks' %run: it runs another notebook's code cells in
# this notebook's namespace. It is used instead of Jupyter's %run because that breaks on
# Windows paths with spaces.
import json
import pathlib

PIPELINE_ROOT = next(
    folder
    for folder in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (folder / "00_shared").is_dir()
)


def run_notebook(relative_path):
    notebook = json.loads((PIPELINE_ROOT / relative_path).read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())


if "SHARED_LOADED" not in globals():
    run_notebook("00_shared/000_nb_configurations.ipynb")
    run_notebook("00_shared/001_nb_utils.ipynb")
    SHARED_LOADED = True

## 1. ctrl.data_flow_control

In [ ]:
import shutil

# Upgrade: a control table from an earlier version of the framework lacks the incremental
# columns. It only holds flow definitions (050 reloads them), so it is safe to recreate.
if is_table_folder(ctrl_table_path("data_flow_control")):
    existing_columns = read_fresh(f"{CTRL_DATABASE}.data_flow_control").columns
    if "watermark_column" not in existing_columns:
        spark.sql(f"DROP TABLE IF EXISTS {CTRL_DATABASE}.data_flow_control")
        shutil.rmtree(ctrl_table_path("data_flow_control"))
        print("Old data_flow_control found: recreated with the incremental columns. Run 050 next.")

ctrl_table_path("data_flow_control").mkdir(parents=True, exist_ok=True)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CTRL_DATABASE}.data_flow_control
(
    -- Identity and control
    control_id              BIGINT,
    pipeline_name           STRING,
    layer                   STRING,     -- BRONZE / SILVER / GOLD
    is_active               BOOLEAN,
    load_sequence           INT,
    depends_on_control_ids  STRING,     -- comma-separated control_ids that must succeed first

    -- Source
    source_system           STRING,
    source_type             STRING,     -- dispatch key into ctrl.source_type_registry
    source_connection_name  STRING,     -- folder key in CONNECTIONS (Bronze only)
    source_object           STRING,     -- file (Bronze) or script (Silver/Gold)
    source_format           STRING,

    -- Destination
    destination_schema      STRING,     -- business schema, e.g. wahat
    destination_table       STRING,

    -- Load behaviour
    load_type               STRING,     -- FULL / INCREMENTAL / PARTITION_REFRESH / SCD2 / CHECK
    primary_key_columns     STRING,     -- MERGE key (INCREMENTAL), business key (SCD2)
    publish_name            STRING,     -- Gold: parquet file name for Power BI

    -- Resilience
    retry_count             INT,
    timeout_minutes         INT,

    -- Incremental behaviour
    watermark_column        STRING,     -- compared between source and target, e.g. GLLineID, _batch_id
    incremental_source      STRING,     -- layer.table read as {{increment}}, e.g. bronze.gl_lines
    partition_column        STRING,     -- PARTITION_REFRESH: target column of the rebuilt range
    partition_source_column STRING,     -- PARTITION_REFRESH: expression on the new source rows
    surrogate_key_column    STRING      -- SCD2: surrogate key added by the loader
)
USING {TABLE_FORMAT}
LOCATION '{ctrl_table_path("data_flow_control").as_posix()}'
""")
print(f"{CTRL_DATABASE}.data_flow_control ready")

## 2. ctrl.source_type_registry

In [ ]:
ctrl_table_path("source_type_registry").mkdir(parents=True, exist_ok=True)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CTRL_DATABASE}.source_type_registry
(
    source_type     STRING,
    layer           STRING,     -- scopes each orchestrator to its own rows
    notebook_name   STRING,     -- loader notebook in that layer's loader folder
    is_active       BOOLEAN,
    description     STRING
)
USING {TABLE_FORMAT}
LOCATION '{ctrl_table_path("source_type_registry").as_posix()}'
""")
print(f"{CTRL_DATABASE}.source_type_registry ready")